<a href="https://colab.research.google.com/github/Myria255/BOOTCAMP-TTA/blob/main/W6D5_Daily_Challenge_Trustworthy_Insights_BERT.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>


# Daily Challenge — Building Trustworthy Insights with BERT

**Developers Institute & Sira Labs — Week 6, Day 5**

## Contexte

L’objectif est de construire un assistant de classification de sentiments capable
de fournir à la fois :

- une prédiction ;
- un score de confiance ;
- des tokens mis en évidence pour aider à interpréter la décision.

## Ce notebook réalise

1. le chargement et l’inspection de `tweet_eval/sentiment` ;
2. la tokenisation des tweets avec DistilBERT ;
3. le fine-tuning du modèle pendant trois époques ;
4. l’évaluation avec l’accuracy et le macro F1 ;
5. l’analyse de la distribution des scores de confiance ;
6. la visualisation de l’attention de `[CLS]` ;
7. la création d’une fonction `analyze_text()` prête à être réutilisée ;
8. la sauvegarde du modèle, du tokenizer et des exemples.

> Dans Google Colab, activez un GPU avec  
> **Exécution → Modifier le type d’exécution → T4 GPU**.


In [ ]:

# Installation de versions compatibles
%pip install -q "transformers==4.48.3" "datasets==3.2.0" \
    "accelerate==1.3.0" "evaluate==0.4.3" \
    "scikit-learn==1.6.1"


In [ ]:

# Imports et configuration
import json
import os
import platform
import random
from collections import Counter

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import torch
import transformers
import datasets

from datasets import DatasetDict, load_dataset
from sklearn.metrics import accuracy_score, f1_score, classification_report
from transformers import (
    AutoModel,
    AutoModelForSequenceClassification,
    AutoTokenizer,
    DataCollatorWithPadding,
    Trainer,
    TrainingArguments,
)

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

ARTIFACT_DIR = "/content/trustworthy_sentiment_artifacts"
MODEL_DIR = os.path.join(ARTIFACT_DIR, "distilbert_tweet_sentiment")
ENCODER_DIR = os.path.join(ARTIFACT_DIR, "distilbert_encoder")
EXAMPLES_PATH = os.path.join(ARTIFACT_DIR, "saved_examples.json")

os.makedirs(ARTIFACT_DIR, exist_ok=True)

print("Python       :", platform.python_version())
print("PyTorch      :", torch.__version__)
print("Transformers :", transformers.__version__)
print("Datasets     :", datasets.__version__)
print("Device       :", DEVICE)

if DEVICE.type == "cpu":
    print(
        "\nAttention : aucun GPU n'est détecté. "
        "L'entraînement sera nettement plus lent sur CPU."
    )



## 1. Chargement et inspection des données

Le sous-ensemble `sentiment` de `tweet_eval` contient trois classes :

- `0` : négatif ;
- `1` : neutre ;
- `2` : positif.

Nous affichons les tailles et la distribution des classes, puis nous sauvegardons
deux exemples par classe pour les visualisations ultérieures.


In [ ]:

# Chargement des splits train, validation et test
raw_dataset = load_dataset(
    "cardiffnlp/tweet_eval",
    "sentiment",
)

print(raw_dataset)

label_names = raw_dataset["train"].features["label"].names
id2label = {index: label for index, label in enumerate(label_names)}
label2id = {label: index for index, label in id2label.items()}

print("\nLabels :", id2label)
assert len(label_names) == 3, "Le dataset doit contenir exactement trois classes."

# Distribution des classes
distribution_rows = []

for split_name, split_dataset in raw_dataset.items():
    counts = Counter(split_dataset["label"])

    for label_id, label_name in id2label.items():
        distribution_rows.append(
            {
                "split": split_name,
                "label_id": label_id,
                "label": label_name,
                "count": counts[label_id],
                "percentage": 100 * counts[label_id] / len(split_dataset),
            }
        )

distribution_df = pd.DataFrame(distribution_rows)

print("\nDistribution des classes :")
display(
    distribution_df.style.format(
        {"percentage": "{:.2f}%"}
    )
)

# Sauvegarde de deux exemples par classe
saved_examples = {}

for label_id, label_name in id2label.items():
    label_subset = raw_dataset["train"].filter(
        lambda example: example["label"] == label_id
    )

    saved_examples[label_name] = [
        {
            "text": label_subset[index]["text"],
            "label": label_id,
        }
        for index in range(min(2, len(label_subset)))
    ]

with open(EXAMPLES_PATH, "w", encoding="utf-8") as file:
    json.dump(saved_examples, file, ensure_ascii=False, indent=2)

print("\nDeux exemples sauvegardés par classe :")
for label_name, examples in saved_examples.items():
    print(f"\n{label_name.upper()}")
    for example in examples:
        print("-", example["text"])

print("\nFichier créé :", EXAMPLES_PATH)



## 2. Pipeline de tokenisation

Le tokenizer de `distilbert-base-uncased` transforme chaque tweet en :

- `input_ids` ;
- `attention_mask` ;
- `label`.

Les séquences sont tronquées à 128 tokens. Le remplissage est effectué
dynamiquement par `DataCollatorWithPadding`, ce qui évite de remplir inutilement
tous les tweets jusqu’à la longueur maximale.


In [ ]:

MODEL_NAME = "distilbert-base-uncased"
MAX_LENGTH = 128

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

def preprocess_function(batch):
    return tokenizer(
        batch["text"],
        truncation=True,
        max_length=MAX_LENGTH,
    )

tokenized_dataset = raw_dataset.map(
    preprocess_function,
    batched=True,
    remove_columns=["text"],
    desc="Tokenisation des tweets",
)

# Mélange reproductible de chaque split
tokenized_dataset = DatasetDict(
    {
        split_name: split_dataset.shuffle(seed=SEED)
        for split_name, split_dataset in tokenized_dataset.items()
    }
)

tokenized_dataset.set_format(
    type="torch",
    columns=["input_ids", "attention_mask", "label"],
)

data_collator = DataCollatorWithPadding(
    tokenizer=tokenizer,
    return_tensors="pt",
)

print(tokenized_dataset)

sample = tokenized_dataset["train"][0]
print("\nExemple tokenisé :")
print("input_ids      :", sample["input_ids"].shape)
print("attention_mask :", sample["attention_mask"].shape)
print("label          :", int(sample["label"]))
print("tokens         :", tokenizer.convert_ids_to_tokens(
    sample["input_ids"][:20].tolist()
))



## 3. Configuration et fine-tuning de DistilBERT

Le modèle est entraîné avec les paramètres demandés :

- 3 époques ;
- batch size de 32 ;
- taux d’apprentissage de `5e-5` ;
- weight decay de `0.01`.

Le meilleur checkpoint est sélectionné selon le **macro F1**, une métrique
adaptée lorsque l’on souhaite accorder la même importance aux trois classes.


In [ ]:

model = AutoModelForSequenceClassification.from_pretrained(
    MODEL_NAME,
    num_labels=3,
    id2label=id2label,
    label2id=label2id,
)

def compute_metrics(eval_prediction):
    logits, labels = eval_prediction
    predictions = np.argmax(logits, axis=-1)

    return {
        "accuracy": accuracy_score(labels, predictions),
        "f1": f1_score(
            labels,
            predictions,
            average="macro",
            zero_division=0,
        ),
    }

training_args = TrainingArguments(
    output_dir=os.path.join(ARTIFACT_DIR, "checkpoints"),
    eval_strategy="epoch",
    save_strategy="epoch",
    load_best_model_at_end=True,
    metric_for_best_model="f1",
    greater_is_better=True,
    save_total_limit=2,
    learning_rate=5e-5,
    per_device_train_batch_size=32,
    per_device_eval_batch_size=32,
    num_train_epochs=3,
    weight_decay=0.01,
    warmup_ratio=0.1,
    logging_steps=100,
    report_to="none",
    fp16=torch.cuda.is_available(),
    seed=SEED,
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_dataset["train"],
    eval_dataset=tokenized_dataset["validation"],
    processing_class=tokenizer,
    data_collator=data_collator,
    compute_metrics=compute_metrics,
)

train_result = trainer.train()

print("\nEntraînement terminé.")
print("Perte moyenne :", train_result.training_loss)
print("Meilleur checkpoint :", trainer.state.best_model_checkpoint)
print("Meilleur macro F1   :", trainer.state.best_metric)


In [ ]:

# Sauvegarde des artefacts finaux
trainer.save_model(MODEL_DIR)
tokenizer.save_pretrained(MODEL_DIR)

# Sauvegarde séparée de l'encodeur pour l'inspection avec AutoModel
trainer.model.distilbert.save_pretrained(ENCODER_DIR)
tokenizer.save_pretrained(ENCODER_DIR)

print("Modèle de classification :", MODEL_DIR)
print("Encodeur DistilBERT       :", ENCODER_DIR)
print("Tokenizer                 :", MODEL_DIR)



## 4. Évaluation et analyse de la confiance

Nous mesurons l’accuracy et le macro F1 sur la validation, puis sur le test.

Pour le test, nous récupérons la probabilité softmax de la classe prédite.
L’histogramme indique si le modèle produit surtout des scores modérés ou très
élevés.

Nous comparons également la confiance moyenne à l’accuracy :

- confiance nettement supérieure à l’accuracy : tendance à la surconfiance ;
- confiance nettement inférieure : tendance à la sous-confiance ;
- valeurs proches : calibration globale raisonnable.


In [ ]:

# Évaluation sur la validation
validation_metrics = trainer.evaluate(
    tokenized_dataset["validation"],
    metric_key_prefix="validation",
)

print("Résultats de validation :")
for metric_name, metric_value in validation_metrics.items():
    if isinstance(metric_value, (float, int)):
        print(f"{metric_name:30s}: {metric_value:.4f}")

# Prédictions sur le test
test_output = trainer.predict(
    tokenized_dataset["test"],
    metric_key_prefix="test",
)

test_logits = test_output.predictions
test_labels = test_output.label_ids
test_probabilities = torch.softmax(
    torch.tensor(test_logits),
    dim=-1,
).numpy()

test_predictions = np.argmax(test_probabilities, axis=-1)
test_confidences = np.max(test_probabilities, axis=-1)
test_correct = test_predictions == test_labels

test_accuracy = accuracy_score(test_labels, test_predictions)
test_macro_f1 = f1_score(
    test_labels,
    test_predictions,
    average="macro",
    zero_division=0,
)

print("\nRésultats de test :")
print(f"Accuracy : {test_accuracy:.4f}")
print(f"Macro F1 : {test_macro_f1:.4f}")

print("\nRapport par classe :")
print(
    classification_report(
        test_labels,
        test_predictions,
        target_names=label_names,
        digits=4,
        zero_division=0,
    )
)


In [ ]:

# Histogramme des scores de confiance
bins = np.arange(0.0, 1.01, 0.1)

plt.figure(figsize=(9, 5))
plt.hist(
    test_confidences,
    bins=bins,
    edgecolor="black",
)
plt.title("Distribution des scores de confiance sur le test")
plt.xlabel("Confiance de la classe prédite")
plt.ylabel("Nombre de tweets")
plt.xticks(bins)
plt.grid(axis="y", alpha=0.3)
plt.show()

mean_confidence = float(np.mean(test_confidences))
confidence_gap = mean_confidence - test_accuracy

print(f"Confiance moyenne : {mean_confidence:.4f}")
print(f"Accuracy test     : {test_accuracy:.4f}")
print(f"Écart             : {confidence_gap:+.4f}")

if confidence_gap > 0.03:
    calibration_comment = (
        "Le modèle présente une tendance globale à la surconfiance : "
        "sa confiance moyenne dépasse sensiblement son accuracy."
    )
elif confidence_gap < -0.03:
    calibration_comment = (
        "Le modèle présente une tendance globale à la sous-confiance : "
        "ses prédictions sont en moyenne plus exactes que ne le suggèrent "
        "ses scores de confiance."
    )
else:
    calibration_comment = (
        "La confiance moyenne et l'accuracy sont relativement proches. "
        "La calibration globale paraît raisonnable, mais une analyse par "
        "intervalle reste recommandée."
    )

print("\nCommentaire de calibration :")
print(calibration_comment)


In [ ]:

# Tableau de calibration par intervalles de 0,1
calibration_rows = []

for lower_bound in np.arange(0.0, 1.0, 0.1):
    upper_bound = lower_bound + 0.1

    if upper_bound >= 1.0:
        mask = (
            (test_confidences >= lower_bound)
            & (test_confidences <= upper_bound)
        )
    else:
        mask = (
            (test_confidences >= lower_bound)
            & (test_confidences < upper_bound)
        )

    count = int(mask.sum())

    if count == 0:
        continue

    calibration_rows.append(
        {
            "intervalle": f"[{lower_bound:.1f}, {upper_bound:.1f}]",
            "nombre": count,
            "confiance_moyenne": float(test_confidences[mask].mean()),
            "accuracy_observee": float(test_correct[mask].mean()),
        }
    )

calibration_df = pd.DataFrame(calibration_rows)
display(
    calibration_df.style.format(
        {
            "confiance_moyenne": "{:.3f}",
            "accuracy_observee": "{:.3f}",
        }
    )
)



## 5. Inspection de l’attention

Nous chargeons l’encodeur fine-tuné avec `AutoModel` et activons
`output_attentions=True`.

Pour un tweet sauvegardé, nous :

1. récupérons l’attention de la dernière couche ;
2. faisons la moyenne sur les têtes ;
3. sélectionnons l’attention allant de `[CLS]` vers chaque token ;
4. affichons ces poids sous forme de barres.

> **Limite importante :** un poids d’attention élevé ne prouve pas qu’un token
> a causé la prédiction. Il constitue seulement un indice d’interprétation.


In [ ]:

# Chargement de l'encodeur fine-tuné pour l'analyse d'attention
attention_model = AutoModel.from_pretrained(
    ENCODER_DIR,
    output_attentions=True,
).to(DEVICE)

attention_model.eval()

# Choix d'un exemple négatif sauvegardé
chosen_example = saved_examples["negative"][0]["text"]

attention_inputs = tokenizer(
    chosen_example,
    return_tensors="pt",
    truncation=True,
    max_length=MAX_LENGTH,
).to(DEVICE)

with torch.no_grad():
    attention_outputs = attention_model(
        **attention_inputs,
        output_attentions=True,
        return_dict=True,
    )

# Forme : [batch, heads, sequence, sequence]
last_layer_attention = attention_outputs.attentions[-1]

# Moyenne sur les têtes, puis attention de [CLS] vers chaque token
mean_attention = last_layer_attention.mean(dim=1)
cls_attention = mean_attention[0, 0, :].detach().cpu().numpy()

input_ids = attention_inputs["input_ids"][0].detach().cpu()
attention_mask = attention_inputs["attention_mask"][0].detach().cpu().numpy()

valid_length = int(attention_mask.sum())
tokens = tokenizer.convert_ids_to_tokens(
    input_ids[:valid_length].tolist()
)
cls_scores = cls_attention[:valid_length]

print("Tweet analysé :")
print(chosen_example)

plt.figure(figsize=(max(10, len(tokens) * 0.45), 5))
plt.bar(range(len(tokens)), cls_scores)
plt.title("Attention moyenne de [CLS] vers les tokens — dernière couche")
plt.xlabel("Tokens")
plt.ylabel("Poids d'attention")
plt.xticks(
    range(len(tokens)),
    tokens,
    rotation=60,
    ha="right",
)
plt.tight_layout()
plt.show()

# Tokens les plus observés, hors tokens spéciaux
special_tokens = set(tokenizer.all_special_tokens)

ranked_tokens = sorted(
    [
        (token, float(score))
        for token, score in zip(tokens, cls_scores)
        if token not in special_tokens
    ],
    key=lambda item: item[1],
    reverse=True,
)

print("\nTokens recevant le plus d'attention depuis [CLS] :")
for token, score in ranked_tokens[:8]:
    print(f"{token:20s} {score:.4f}")



### Interprétation de l’exemple

Les tokens placés en tête du classement sont ceux auxquels la représentation
`[CLS]` accorde le plus d’attention dans la dernière couche. Lorsque des termes
fortement sentimentaux apparaissent parmi eux, cela suggère que le modèle les
utilise pour construire sa représentation globale.

Cependant, l’attention peut également se concentrer sur des mots de liaison,
des négations, de la ponctuation ou des sous-mots. L’interprétation doit donc
tenir compte du contexte complet et ne doit pas être considérée comme une
explication causale définitive.



## 6. Fonction d’inférence explicable

`analyze_text()` retourne :

```python
{
    "label": "...",
    "confidence": 0.00,
    "highlighted_tokens": [
        {"token": "...", "attention": 0.00}
    ]
}
```

Les tokens sont classés selon l’attention moyenne reçue depuis `[CLS]` dans la
dernière couche du classifieur fine-tuné.


In [ ]:

# Rechargement du modèle sauvegardé comme dans un contexte de déploiement
inference_model = AutoModelForSequenceClassification.from_pretrained(
    MODEL_DIR,
).to(DEVICE)

inference_model.eval()

def analyze_text(text: str, top_k: int = 5) -> dict:
    """Prédit le sentiment et retourne des tokens indicatifs.

    Attention : les highlighted_tokens sont fondés sur les poids
    d'attention. Ils constituent un indice, pas une preuve causale.
    """

    if not isinstance(text, str) or not text.strip():
        raise ValueError("Le texte doit être une chaîne non vide.")

    if top_k <= 0:
        raise ValueError("top_k doit être strictement positif.")

    encoded = tokenizer(
        text,
        return_tensors="pt",
        truncation=True,
        max_length=MAX_LENGTH,
    ).to(DEVICE)

    with torch.no_grad():
        outputs = inference_model(
            **encoded,
            output_attentions=True,
            return_dict=True,
        )

    probabilities = torch.softmax(outputs.logits, dim=-1)[0]
    predicted_id = int(torch.argmax(probabilities).item())
    confidence = float(probabilities[predicted_id].item())

    # Attention de dernière couche : moyenne des têtes
    last_attention = outputs.attentions[-1]
    mean_attention = last_attention.mean(dim=1)
    cls_to_tokens = mean_attention[0, 0, :].detach().cpu().numpy()

    ids = encoded["input_ids"][0].detach().cpu().tolist()
    mask = encoded["attention_mask"][0].detach().cpu().tolist()
    valid_length = int(sum(mask))

    tokens = tokenizer.convert_ids_to_tokens(ids[:valid_length])
    scores = cls_to_tokens[:valid_length]
    special_tokens = set(tokenizer.all_special_tokens)

    candidates = [
        {
            "token": token,
            "attention": float(score),
        }
        for token, score in zip(tokens, scores)
        if token not in special_tokens
    ]

    candidates.sort(
        key=lambda item: item["attention"],
        reverse=True,
    )

    return {
        "label": id2label[predicted_id],
        "confidence": confidence,
        "highlighted_tokens": candidates[:top_k],
    }


example_texts = [
    "The service was terrible and nobody helped me.",
    "The update is available tomorrow morning.",
    "I absolutely love the new design!",
]

for example_text in example_texts:
    result = analyze_text(example_text, top_k=5)

    print("\nTexte :", example_text)
    print(json.dumps(result, indent=2, ensure_ascii=False))



## 7. Préparation au déploiement

Les artefacts suivants ont été sauvegardés :

- le modèle de classification fine-tuné ;
- le tokenizer ;
- l’encodeur DistilBERT destiné à l’inspection ;
- deux exemples par classe ;
- les checkpoints d’entraînement.

### Garde-fous recommandés

Avant une intégration dans un outil de support :

- définir un seuil de confiance ;
- transmettre les cas incertains à un humain ;
- contrôler les erreurs par classe ;
- surveiller la dérive des tweets ou messages réels ;
- ne pas traiter l’attention comme une explication causale ;
- documenter la version du modèle et du dataset ;
- protéger les données personnelles.

## Conclusion

Le projet associe performance, confiance et interprétabilité. DistilBERT fournit
une classification légère, le macro F1 mesure l’équilibre entre les classes et
l’analyse d’attention donne des indices sur les tokens observés par le modèle.
La fonction `analyze_text()` peut être intégrée dans un outil interne, à
condition de conserver une supervision humaine et un suivi régulier.



## Soumission

1. Exécuter toutes les cellules avec un GPU T4.
2. Conserver les sorties, graphiques et tableaux dans le notebook.
3. Enregistrer une copie dans Google Drive.
4. Partager le notebook avec l’option **Tous les utilisateurs disposant du lien**.
5. Facultatif : enregistrer une copie dans GitHub.
6. Soumettre le lien public sur la plateforme DI.
